# 🏥 HealthBridge Medical Center
## Notebook 02 — Exploratory Data Analysis (EDA)

**Load:** `diabetic_clean.csv` (output from Notebook 01)

**Goal:** Understand distributions, spot patterns, visualize readmission rates by key variables.

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('diabetic_clean.csv')
print(f'Shape: {df.shape}')
df.head()

## Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original 3-class
df['readmitted'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#2E75B6','#1D7A4F','#C00000'], edgecolor='black', rot=0)
axes[0].set_title('Original readmitted column (3 classes)', fontweight='bold')
axes[0].set_ylabel('Patient Count')
for bar in axes[0].patches:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                 f'{bar.get_height():,.0f}', ha='center', fontsize=10)

# Binary target
binary = (df['readmitted']=='<30').astype(int)
binary.value_counts().rename({0:'Not readmitted (0)', 1:'Readmitted <30 days (1)'}).plot(
    kind='bar', ax=axes[1], color=['#2E75B6','#C00000'], edgecolor='black', rot=0)
axes[1].set_title('Binary target: readmitted_30', fontweight='bold')
axes[1].set_ylabel('Patient Count')
plt.suptitle('HealthBridge — Target Variable Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Positive class (readmitted <30): {binary.mean()*100:.1f}%')

## Readmission Rate by Age Group

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
binary = (df['readmitted']=='<30').astype(int)
df['readmitted_30'] = binary

df['age'].value_counts().sort_index().plot(kind='bar', ax=axes[0],
    color='#2E75B6', edgecolor='black', rot=45)
axes[0].set_title('Patient Count by Age Group', fontweight='bold')
axes[0].set_xlabel('Age Group')

readmit_age = df.groupby('age')['readmitted_30'].mean().mul(100)
readmit_age.plot(kind='bar', ax=axes[1], color='#C00000', edgecolor='black', rot=45)
axes[1].axhline(binary.mean()*100, color='navy', linestyle='--',
                label=f'Overall avg: {binary.mean()*100:.1f}%')
axes[1].set_title('30-Day Readmission Rate by Age', fontweight='bold')
axes[1].set_ylabel('Readmission Rate (%)')
axes[1].legend()
plt.tight_layout()
plt.savefig('reports/figures/readmission_by_age.png', dpi=150, bbox_inches='tight')
plt.show()

## Readmission Rate by Prior Inpatient Visits

In [ ]:
df['inpatient_capped'] = df['number_inpatient'].clip(upper=5)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df['inpatient_capped'].value_counts().sort_index().plot(kind='bar', ax=axes[0],
    color='#1D7A4F', edgecolor='black', rot=0)
axes[0].set_title('Patient Count by Prior Inpatient Visits', fontweight='bold')
axes[0].set_xlabel('Prior Inpatient Visits (Past Year)')

df.groupby('inpatient_capped')['readmitted_30'].mean().mul(100).plot(
    kind='bar', ax=axes[1], color='#C00000', edgecolor='black', rot=0)
axes[1].axhline(binary.mean()*100, color='navy', linestyle='--', label='Overall avg')
axes[1].set_title('Readmission Rate by Prior Visits', fontweight='bold')
axes[1].set_ylabel('Readmission Rate (%)')
axes[1].legend()
plt.tight_layout()
plt.savefig('reports/figures/readmission_by_prior_visits.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: Patients with more prior visits are far more likely to be readmitted')

## Correlation Heatmap — Numeric Features

In [ ]:
num_cols = ['time_in_hospital','num_lab_procedures','num_procedures',
            'num_medications','number_outpatient','number_emergency',
            'number_inpatient','number_diagnoses','readmitted_30']
corr = df[num_cols].corr()
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Numeric Features', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('reports/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()